Make sure top left says `traffic` and not `Select Kernel`

In [ ]:
!uv pip install kaggle

In [ ]:
import os
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import zipfile

In [ ]:
subprocess.run(["kaggle", "datasets", "download", "-d", "liuxu77/largest", "-p", "../data/external/LargeST/data/ca/", "--unzip"])

In [ ]:
# Remove the zip file if it exists
zip_path = Path("../data/external/LargeST/data/ca/largest.zip")
if zip_path.exists():
    os.remove(zip_path)

# Remove the following files if they exist
files_to_remove = [
    Path("../data/external/LargeST/data/ca/ca_his_raw_2020.h5"),
    Path("../data/external/LargeST/data/ca/ca_his_raw_2021.h5"),
]
for file_path in files_to_remove:
    if file_path.exists():
        os.remove(file_path)

In [ ]:
os.makedirs('../data/external/LargeST/data/sd/largest/', exist_ok=True)

In [ ]:
ca_meta = pd.read_csv('../data/external/LargeST/data/ca/ca_meta.csv')
sd_meta = ca_meta[ca_meta.District == 11]
sd_meta = sd_meta.reset_index()
sd_meta = sd_meta.drop(columns=['index'])
sd_meta.to_csv('../data/external/LargeST/data/sd/largest/sd_meta.csv', index=False)
print(sd_meta[sd_meta.duplicated(subset=['Lat', 'Lng'])])
sd_meta

In [ ]:
sd_meta_id2 = sd_meta.ID2.values.tolist()
print(len(sd_meta_id2))

ca_rn_adj = np.load('../data/external/LargeST/data/ca/ca_rn_adj.npy')
print(ca_rn_adj.shape)

sd_rn_adj = ca_rn_adj[sd_meta_id2]
sd_rn_adj = sd_rn_adj[:,sd_meta_id2]
print(sd_rn_adj.shape)

np.save('../data/external/LargeST/data/ca/sd_rn_adj.npy', sd_rn_adj)

In [ ]:
years = ['2017', '2018', '2019']

sd_meta.ID = sd_meta.ID.astype(str)
sd_meta_id = sd_meta.ID.values.tolist()

for year in years:
    ca_his = pd.read_hdf('../data/external/LargeST/data/ca/ca_his_raw_' + year +'.h5')
    sd_his = ca_his[sd_meta_id]
    sd_his.to_hdf('../data/external/LargeST/data/sd/largest/sd_his_' + year + '.h5', key='t', mode='w')

In [ ]:
os.makedirs('../data/pollution/no2', exist_ok=True)
os.makedirs('../data/pollution/co', exist_ok=True)
os.makedirs('../data/pollution/pm25', exist_ok=True)

In [ ]:
env_sources = {
    "no2" : [
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42602_2017.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42602_2018.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42602_2019.zip",
    ],
    "co" : [
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42101_2017.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42101_2018.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_42101_2019.zip",
    ],
    "pm25" : [
        "https://aqs.epa.gov/aqsweb/airdata/hourly_88101_2017.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_88101_2018.zip",
        "https://aqs.epa.gov/aqsweb/airdata/hourly_88101_2019.zip",
    ]
}

for env, sources in env_sources.items():
    for source in sources:
        filename = source.split("/")[-1]
        save_path = f"../data/pollution/{env}/{filename}"
        subprocess.run(["curl", "-L", "-o", save_path, source])

        # Unzip the file
        with zipfile.ZipFile(save_path, 'r') as zip_ref:
            zip_ref.extractall(f"../data/pollution/{env}/")
        # Remove the zip file
        if os.path.exists(save_path):
            os.remove(save_path)
        